# Ball possession — Colab (Google Drive only)

**You only need 3 files on Drive.** No git clone. No videos. Homography is optional and **not used** here.


## Idea of the solution 1 (simple)

We answer one question every frame: **who has the ball?**

1. **Look at the ball and the players**  
   When tracking can see the ball, find the **closest player** to it.  
   That player’s team gets the ball (`H` = Barcelona, `A` = Athletic).

2. **When the ball disappears for a moment**  
   Tracking loses the ball a lot (about 40% of the time).  
   We **keep the last team** we saw, instead of guessing a new one.  
   Missing ball ≠ dead ball.

3. **Mark stoppages as dead**  
   Smart Tagging tells us when there is a **set piece** (corner, throw-in, free kick, etc.).  
   During those moments we label **`D`** (dead ball / not in play).

4. **Use tactical phases as a backup**  
   If we still don’t know the team, we use Smart Tagging’s “which team is attacking / building up” tag as a hint.

5. **Smooth the answer**  
   Frame-by-frame answers flicker (H–A–H–A).  
   We take a **majority vote over ~2 seconds** around each frame so possessions stay as longer, cleaner stretches.  
   That matters because the contest ranks **whole possessions**, not single frames.

6. **Outside the match**  
   Before kickoff, at half-time, after full time → we write **`X`** (not scored).

**What we do not use:** the video, or the homography. Positions already come from tracking.

In [ ]:
# ========= ONLY THING YOU MAY NEED TO EDIT =========
DRIVE_ROOT = "/content/drive/MyDrive/Game Theory Barcelona"
# ====================================================

DATA_DIR = f"{DRIVE_ROOT}/data"
TACTICAL = f"{DATA_DIR}/tactical"
OUT_DIR = f"{DRIVE_ROOT}/submissions"

ATD_XML = f"{TACTICAL}/BAR-ATH_tactical_atd.xml"
ATD_TXT = f"{TACTICAL}/BAR-ATH_tactical_atd.txt"
SMART_XML = f"{TACTICAL}/BAR-ATH_tactical_smart_tagging.xml"

N_FRAMES = 199319
FPS = 25
PITCH = (105.3, 68.0)  # metres
# Graded halves (published boundaries)
HALF_BLOCKS = [(1, 11185, 90651), (2, 110808, 190197)]

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os, sys, subprocess
import xml.etree.ElementTree as ET
import numpy as np
import pandas as pd

os.makedirs(OUT_DIR, exist_ok=True)
!pip -q install numpy pandas

needed = {
    "atd xml": ATD_XML,
    "atd txt": ATD_TXT,
    "smart tagging": SMART_XML,
}
ok = True
for name, path in needed.items():
    exists = os.path.isfile(path)
    print(("OK  " if exists else "MISSING"), name, "->", path)
    ok = ok and exists

assert ok, (
    "Upload the 3 files under:\n"
    f"  {TACTICAL}/\n"
    "BAR-ATH_tactical_atd.xml\n"
    "BAR-ATH_tactical_atd.txt\n"
    "BAR-ATH_tactical_smart_tagging.xml"
)
print("\nData OK — nothing else required.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
OK   atd xml -> /content/drive/MyDrive/Game Theory Barcelona/data/tactical/BAR-ATH_tactical_atd.xml
OK   atd txt -> /content/drive/MyDrive/Game Theory Barcelona/data/tactical/BAR-ATH_tactical_atd.txt
OK   smart tagging -> /content/drive/MyDrive/Game Theory Barcelona/data/tactical/BAR-ATH_tactical_smart_tagging.xml

Data OK — nothing else required.


## Build labels

Nearest player to ball + fill gaps + Smart Tagging set pieces as dead + smooth.

In [ ]:
HOME, AWAY, NOBODY = 0, 1, -1


def team_of_each_track(meta_path):
    root = ET.parse(meta_path).getroot()
    team_ids = [t.get("id") for t in root.iter("Team")]
    home_id = team_ids[0]
    slots = [p.get("teamId") for p in root.iter("Player")]
    return np.array([HOME if tid == home_id else AWAY for tid in slots], dtype=np.int8)


def nearest_player_codes(raw_path, team_of_track, n_frames, pitch):
    length, width = pitch
    out = np.full(n_frames, NOBODY, dtype=np.int8)
    with open(raw_path, encoding="utf-8", errors="replace") as fh:
        for line in fh:
            parts = line.split(":")
            if len(parts) != 3:
                continue
            ball = parts[2]
            if ball[0] == "N":
                continue
            c1 = ball.index(",")
            c2 = ball.index(",", c1 + 1)
            bx = float(ball[:c1]) * length
            by = float(ball[c1 + 1:c2]) * width
            best_d2, best_team = None, NOBODY
            for slot, group in enumerate(parts[1].split(";")):
                if not group or group[0] == "N":
                    continue
                g1 = group.index(",")
                g2 = group.index(",", g1 + 1)
                dx = float(group[:g1]) * length - bx
                dy = float(group[g1 + 1:g2]) * width - by
                d2 = dx * dx + dy * dy
                if best_d2 is None or d2 < best_d2:
                    best_d2, best_team = d2, int(team_of_track[slot])
            out[int(parts[0]) - 1] = best_team
    return out


print("Reading tracking (1–2 min)...")
codes = nearest_player_codes(ATD_TXT, team_of_each_track(ATD_XML), N_FRAMES, PITCH)
print(f"ball seen on {(codes != NOBODY).sum()} / {N_FRAMES} frames")

Reading tracking (1–2 min)...
ball seen on 113488 / 199319 frames


In [ ]:
def load_smart_tagging(path, fps=25):
    events = []
    for inst in ET.parse(path).iter("instance"):
        start_s = float(inst.findtext("start"))
        end_s = float(inst.findtext("end"))
        events.append({
            "code": (inst.findtext("code") or "").strip(),
            "start_frame": int(round(start_s * fps)),
            "end_frame": int(round(end_s * fps)),
            "tags": {lb.findtext("group"): lb.findtext("text")
                     for lb in inst.findall("label")},
        })
    return events


def team_tag_to_code(team_str):
    if not team_str:
        return NOBODY
    t = team_str.strip().lower()
    if "barcel" in t:
        return HOME
    if "atlet" in t or "athletic" in t:
        return AWAY
    return NOBODY


events = load_smart_tagging(SMART_XML, FPS)

dead = np.zeros(N_FRAMES, dtype=bool)
DEAD_PRE_BUFFER = 25  # 1s before set piece
for ev in events:
    if ev["code"].upper() != "SET PIECES":
        continue
    a = max(0, ev["start_frame"] - DEAD_PRE_BUFFER)
    b = min(N_FRAMES, ev["end_frame"])
    dead[a:b] = True

phase_prior = np.full(N_FRAMES, NOBODY, dtype=np.int8)
for ev in events:
    if ev["code"].upper() == "SET PIECES":
        continue
    tc = team_tag_to_code(ev["tags"].get("Team"))
    if tc == NOBODY:
        continue
    a = max(0, ev["start_frame"])
    b = min(N_FRAMES, ev["end_frame"])
    phase_prior[a:b] = tc

print(f"set-piece dead frames: {dead.sum()}")
print(f"phase prior frames: {(phase_prior != NOBODY).sum()}")

set-piece dead frames: 45237
phase prior frames: 129381


In [ ]:
def forward_fill_teams(codes):
    out = codes.copy()
    last = NOBODY
    for i, c in enumerate(out):
        if c in (HOME, AWAY):
            last = c
        elif last != NOBODY:
            out[i] = last
    return out


def apply_phase_prior(codes, prior):
    out = codes.copy()
    mask = (out == NOBODY) & (prior != NOBODY)
    out[mask] = prior[mask]
    return out


def smooth_ha(codes_ha, window):
    if window <= 0:
        return codes_ha.copy()
    home_c = np.concatenate(([0], np.cumsum(codes_ha == HOME)))
    away_c = np.concatenate(([0], np.cumsum(codes_ha == AWAY)))
    idx = np.arange(len(codes_ha))
    lo = np.maximum(0, idx - window)
    hi = np.minimum(len(codes_ha), idx + window + 1)
    h = home_c[hi] - home_c[lo]
    a = away_c[hi] - away_c[lo]
    out = np.full(len(codes_ha), NOBODY, dtype=np.int8)
    seen = (h + a) > 0
    out[seen] = np.where(h[seen] >= a[seen], HOME, AWAY).astype(np.int8)
    return out


def codes_to_labels(live_codes, dead_mask, half_blocks):
    labels = np.full(len(live_codes), "X", dtype="<U1")
    graded = np.zeros(len(live_codes), dtype=bool)
    for _, first, last in half_blocks:
        graded[first:last + 1] = True
    for i in np.where(graded)[0]:
        if dead_mask[i]:
            labels[i] = "D"
        elif live_codes[i] == HOME:
            labels[i] = "H"
        elif live_codes[i] == AWAY:
            labels[i] = "A"
        else:
            labels[i] = "D"
    return labels


def build_submission(nearest_codes, dead_mask, phase_prior, half_blocks, window=50):
    filled = apply_phase_prior(forward_fill_teams(nearest_codes), phase_prior)
    for_smooth = filled.copy()
    for_smooth[dead_mask] = NOBODY
    smoothed = forward_fill_teams(smooth_ha(for_smooth, window))
    return codes_to_labels(smoothed, dead_mask, half_blocks)


def write_submission(path, labels):
    with open(path, "w", encoding="utf-8", newline="\n") as fh:
        fh.write("frame,label\n")
        fh.writelines(f"{i},{lab}\n" for i, lab in enumerate(labels))
    return path


def validate_labels(labels):
    assert len(labels) == N_FRAMES, f"need {N_FRAMES} rows, got {len(labels)}"
    bad = set(labels) - {"H", "A", "D", "X"}
    assert not bad, f"bad labels: {bad}"
    print("validate OK:", len(labels), "rows")
    print(pd.Series(labels).value_counts().sort_index().to_string())

In [ ]:
SMOOTH_WINDOW = 50

labels = build_submission(codes, dead, phase_prior, HALF_BLOCKS, window=SMOOTH_WINDOW)
out_csv = f"{OUT_DIR}/possession_submission.csv"
write_submission(out_csv, labels)
validate_labels(labels)
print("\nSaved to Drive:", out_csv)

validate OK: 199319 rows
A    52330
D    45124
H    61403
X    40462

Saved to Drive: /content/drive/MyDrive/Game Theory Barcelona/submissions/possession_submission.csv


## See how well it performs (public 5 minutes)

This downloads the **public** ground truth + official scorer from GitHub (not your match data).

You get `macroF1` and `spellF1` on the first 5 minutes of play only.

- **spellF1** = what the leaderboard ranks on  
- Published floor on the *full* match ≈ **0.2433** (your 5‑min number will look different — that’s normal)  
- Use this to check the pipeline works — **don’t tune** hard on these 5 minutes

In [ ]:
import urllib.request

SCORE_DIR = "/content/mtah_score"
os.makedirs(SCORE_DIR, exist_ok=True)

BASE = "https://raw.githubusercontent.com/metrica-sports/more-than-a-hack-2027/main"
files_to_get = {
    "possession_metrics.py": f"{BASE}/evaluation/possession_metrics.py",
    "BAR-ATH_possession_gt_public.csv": f"{BASE}/evaluation/gt/BAR-ATH_possession_gt_public.csv",
}

for name, url in files_to_get.items():
    dest = os.path.join(SCORE_DIR, name)
    if not os.path.isfile(dest):
        print("downloading", name, "...")
        urllib.request.urlretrieve(url, dest)
    else:
        print("already have", name)

sys.path.insert(0, SCORE_DIR)
import importlib
import possession_metrics as pm
importlib.reload(pm)

gt = pm.label_codes(pm.read_label_csv(
    os.path.join(SCORE_DIR, "BAR-ATH_possession_gt_public.csv")
))
pred = pm.label_codes(labels)
r = pm.evaluate(gt, pred)

print("\n=== PUBLIC 5-MINUTE SCORE ===")
print(f"macroF1 : {r['macro_f1']:.4f}")
print(f"spellF1 : {r['spell_f1_mean']:.4f}   <- ranking metric")
print(f"your possessions : {r['n_pred_spells']}  (GT has 17)")
print(f"your mean dur   : {r['mean_spell_duration_s']:.1f}s  (GT ~14.2s)")
print("\nReference on same 5 min (official baselines):")
print("  baseline_raw        spellF1 ~0.04")
print("  baseline_smooth50   spellF1 ~0.33")
print("  baseline_smooth125  spellF1 ~0.42")

downloading possession_metrics.py ...
downloading BAR-ATH_possession_gt_public.csv ...

=== PUBLIC 5-MINUTE SCORE ===
macroF1 : 0.6894
spellF1 : 0.3300   <- ranking metric
your possessions : 23  (GT has 17)
your mean dur   : 10.9s  (GT ~14.2s)

Reference on same 5 min (official baselines):
  baseline_raw        spellF1 ~0.04
  baseline_smooth50   spellF1 ~0.33
  baseline_smooth125  spellF1 ~0.42


#Idea 2

# Our approach to the possession challenge, start to finish

## 1. The task
For every frame of Barcelona vs Athletic (25 frames per second, 199,319 frames in total), we write one letter:

| Label | Meaning |
|---|---|
| **H** | Home team (**Barcelona**) has the ball, play is live |
| **A** | Away team (**Athletic Club**) has the ball, play is live |
| **D** | Dead ball: the ball is out of play (throw-in, goal kick, corner, free kick, ball outside the pitch) |
| **X** | No answer. We use it only outside the two graded halves (before kickoff, half-time, after the final whistle), where nothing is scored |

Barcelona is always **H** and Athletic is always **A**, in the code and in the CSV.

---

## 2. How we are scored (spellF1)
A **possession** is a run of the same team letter. `H H H A A` is two possessions. `H H D D H H` is two possessions with a stoppage between them.

spellF1 asks: *did you find the same possessions as the real ones, at about the same times?* It matches each predicted possession to a real one of the same team by how much their time spans overlap.

- **Too many flips (H-A-H-A):** fake possessions. They count as false alarms, and they chop real possessions into pieces, which counts as misses.
- **Too few possessions:** real ones get merged and missed.
- **A constant guess:** scores exactly 0.

So the goal is **the right number of possessions, starting and ending at the right times**, not just the most correct frames.

---

## 3. Idea 1: nearest player, then a vote
1. At each frame, find the player nearest the ball. That player's team has it.
2. If the ball is missing (about 40% of frames), keep the last team.
3. Mark set pieces from Smart Tagging as dead.
4. Smooth with a majority vote over about 2 seconds.

**Why it plateaus:** it looks at one snapshot at a time. If Barcelona plays a long pass and an Athletic defender happens to be nearest mid-flight, we output **A** for a few seconds, then **H** again. That creates a fake Athletic possession.

---

## 4. Idea 2: pick the most believable story for the whole half
Instead of voting frame by frame, we compare complete stories for each half and choose the one with the highest score. This is a **Viterbi decoder** with three states: H, A and D. Every story earns and loses points:

- **Earn points** when a frame agrees with the story (the story says **H** and a Barcelona player is clearly in control of the ball).
- **Lose points** every time the story changes state. This is the price of changing your mind, and we call it `P_SWITCH`.

**Mini example.** A 10-second clip, with one clue per second (positive means Barcelona in control, negative means Athletic), and a switch costs 2 points:

| second | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 | 10 |
|---|---|---|---|---|---|---|---|---|---|---|
| clue | +0.8 | +0.9 | +0.7 | -0.6 | 0 | 0 | +0.8 | +0.9 | +0.8 | +0.7 |

Seconds 4 to 6 are the long pass: a defender is nearest, then the ball vanishes.

- Idea 1 outputs `H H H A A A H H H H`, with a fake possession in the middle.
- Idea 2 compares two stories:
  - Stay **H** all the way: it loses 0.6 points at second 4 and pays no switch.
  - Go to **A** and back: it gains 0.6 points but pays 2 + 2 = 4 for the switches.
  - The first story wins, so the output is all **H**.

A real change of possession has a long run of consistent clues for the other team, and that is enough to pay for a switch. A brief deflection is not. In the real code the switch price is `P_SWITCH = 12`, applied per frame at 25 frames per second.

---

## 5. What counts as a clue
- **Ball control:** a clue is strong when one team's player is close to the ball and clearly closer than the other team. It fades to almost zero when the ball is far from everyone (a long pass in the air).
- **Missing ball (about 40% of frames):** the clue is 0 for every story, so no switch is paid and the last state just carries on. A missing ball never creates a fake possession.
- **Smart Tagging team hints:** small nudges about which team is attacking, and about who restarts after a set piece.

---

## 6. Dead ball (the biggest discovery)
Our first dead-ball rule was "any frame inside a Smart Tagging set-piece note is **D**". We checked the notes and found that **every one lasts exactly 35 seconds** (kick-offs 5 s). A real throw-in takes much less, so we were labelling live attacks as dead. Three real dead stretches in the public window also had no note at all.

So we changed the job of those notes:

> **The notes say where to look. The ball decides if play is dead.**

Inside a note's zone (starting 10 seconds before the note, because note starts are also fuzzy), we give evidence for **D** only when:

- the ball is **missing**, or
- the ball is **outside the pitch**, or
- the ball is **moving very slowly**. The cutoff is the slowest 20% of ball speeds outside the zones in the full match, so it comes from the data and not from the answer key.

If the ball is moving normally inside a zone, play is live and the label stays **H** or **A**.

---

## 7. The pipeline in five steps
1. Read the tracking file. For every frame, compute the distance from the ball to the nearest Barcelona player and the nearest Athletic player, and the ball's speed.
2. Turn that into a clue score for **H** vs **A**.
3. Turn the Smart Tagging notes and the ball's behaviour into dead-ball clues for **D**.
4. Run the Viterbi decoder on each half.
5. Write the CSV, using **X** outside the graded halves.

---

## 8. Results

**Full match (no answer key involved):**

| Version | Possessions | Mean length | Dead share | Dead stretches over 40 s |
|---|---|---|---|---|
| v2 (set-piece notes = dead) | 320 | 14.1 s | 28.9% | 18% |
| v4 (ball decides dead) | 353 | 14.4 s | 19.8% | 5% |

**Our saved file (v4):**
- **H** 72,536 frames (about 48 min), **A** 54,793 (about 36 min), **D** 31,528 (about 21 min), **X** 40,462 (the ungraded frames).
- The file validates with 199,319 rows.

**Public 5-minute check:** spellF1 **0.40** and macroF1 **0.73**, with 20 predicted possessions against 17 real ones. The official baseline scored 0.33 (spellF1) on the same window. That window has only 17 possessions, so one possession moves the score by about 6 points. **We should read this as "the pipeline is correct", not as proof we beat the baseline.** We haven't seen our full-match score.

---

## 9. Where we still lose points
- **Wrong team on about 1 in 5 live frames**, usually across long stretches. We think this is the biggest remaining loss, because each wrong stretch costs a whole possession.
- **Dead stretches with no note and a visible ball** are only about half caught.

---

## 10. Next steps
1. Fill short ball gaps (under half a second) by interpolation before computing the control clue.
2. When the ball is missing, use team shape (who is pushing up, who is dropping back) as extra evidence.
3. **Required for the submission:**
   - a public repository that regenerates our exact CSV, with the match data left out (the rules forbid publishing it);
   - a short technical document that explains the method.

---

## The one-sentence version
> "We stopped asking 'who is nearest right now?' and started asking 'what is the most believable story of this half?', where every change of possession has a price, a missing ball costs nothing, and dead ball is decided by what the ball is actually doing."

In [ ]:
HOME, AWAY, NOBODY = 0, 1, -1

def team_of_each_track(meta_path):
    root = ET.parse(meta_path).getroot()
    team_ids = [t.get("id") for t in root.iter("Team")]
    home_id = team_ids[0]
    slots = [p.get("teamId") for p in root.iter("Player")]
    return np.array([HOME if tid == home_id else AWAY for tid in slots], dtype=np.int8)

def extract_features(raw_path, team_of_track, n_frames, pitch):
    """Per frame: nearest-home distance, nearest-away distance (m), ball position (0-1)."""
    L, W = pitch
    dH = np.full(n_frames, np.nan, np.float32)
    dA = np.full(n_frames, np.nan, np.float32)
    bx = np.full(n_frames, np.nan, np.float32)
    by = np.full(n_frames, np.nan, np.float32)
    with open(raw_path, encoding="utf-8", errors="replace") as fh:
        for line in fh:
            parts = line.split(":")
            if len(parts) != 3:
                continue
            ball = parts[2]
            if ball[0] == "N":
                continue
            f = int(parts[0]) - 1
            c1 = ball.index(","); c2 = ball.index(",", c1 + 1)
            nx, ny = float(ball[:c1]), float(ball[c1 + 1:c2])
            bx[f], by[f] = nx, ny
            x, y = nx * L, ny * W
            best = [np.inf, np.inf]
            for slot, g in enumerate(parts[1].split(";")):
                if not g or g[0] == "N":
                    continue
                g1 = g.index(","); g2 = g.index(",", g1 + 1)
                d2 = (float(g[:g1]) * L - x) ** 2 + (float(g[g1 + 1:g2]) * W - y) ** 2
                t = team_of_track[slot]
                if d2 < best[t]:
                    best[t] = d2
            dH[f] = np.sqrt(best[0]) if np.isfinite(best[0]) else np.nan
            dA[f] = np.sqrt(best[1]) if np.isfinite(best[1]) else np.nan
    return dH, dA, bx, by

print("Reading tracking (1-2 min)...")
dH, dA, bx, by = extract_features(ATD_TXT, team_of_each_track(ATD_XML), N_FRAMES, PITCH)
print("ball seen on", np.isfinite(bx).sum(), "/", N_FRAMES)

Reading tracking (1-2 min)...
ball seen on 113488 / 199319


In [ ]:
# --- 1. Soft "who controls the ball" score in [-1, 1]  (+ = home, - = away)
SIGMA_NEAR = 2.0   # m: how quickly confidence fades as the ball gets far from everyone
SIGMA_MARGIN = 1.5 # m: how much closer the winner must be to count fully

def control_score(dH, dA):
    s = np.zeros(len(dH), np.float32)
    ok = np.isfinite(dH) & np.isfinite(dA)
    near = np.minimum(dH[ok], dA[ok])
    s[ok] = np.tanh((dA[ok] - dH[ok]) / SIGMA_MARGIN) * np.exp(-near / SIGMA_NEAR)
    return s       # ball missing, or flying free of everybody -> ~0 = "no evidence"

ctrl = control_score(dH, dA)

# --- 2. Ball clearly outside the pitch -> dead
oob = np.zeros(N_FRAMES, bool)
seen = np.isfinite(bx)
oob[seen] = (bx[seen] < -0.003) | (bx[seen] > 1.003) | (by[seen] < -0.003) | (by[seen] > 1.003)

# --- 3. Restart prior: the team named on a set piece usually starts the next possession
restart_prior = np.full(N_FRAMES, NOBODY, np.int8)
RESTART_WINDOW = 75   # 3 s after the set piece ends
for ev in events:
    if ev["code"].upper() != "SET PIECES":
        continue
    tc = team_tag_to_code(ev["tags"].get("Team"))   # throw-ins are N/A -> skipped
    if tc == NOBODY:
        continue
    a = min(N_FRAMES, ev["end_frame"]); b = min(N_FRAMES, a + RESTART_WINDOW)
    restart_prior[a:b] = tc

print("OOB frames:", oob.sum(), "| restart-prior frames:", (restart_prior != NOBODY).sum())

OOB frames: 4688 | restart-prior frames: 2025


In [ ]:
# ---- Parameters: set from football logic, NOT tuned on the public 5 minutes ----
W_BALL      = 1.0    # per-frame weight of control evidence
W_PHASE     = 0.15   # weak Smart Tagging "team in phase" hint (every frame)
W_RESTART   = 0.6    # restart-team hint after a set piece
W_SETPIECE  = 3.0    # strong dead-ball evidence from a set piece
W_OOB       = 1.0    # dead-ball evidence from ball outside the pitch
D_BIAS      = -0.3   # dead must be earned: default per-frame cost of the D state
P_SWITCH    = 40.0   # cost of H<->A: ~1.5 s of one-sided control to flip
P_ENTER_D   = 25.0   # cost of entering dead ball
P_EXIT_D    = 8.0    # cost of leaving it (restarts are quick)
MIN_SPELL   = 25     # frames: absorb possessions shorter than 1 s

def build_emissions(lo, hi):
    s = ctrl[lo:hi]; n = hi - lo
    E = np.zeros((n, 3), np.float32)
    pp = phase_prior[lo:hi]; rp = restart_prior[lo:hi]
    E[:, 0] = W_BALL * s + W_PHASE * (pp == HOME) + W_RESTART * (rp == HOME)
    E[:, 1] = -W_BALL * s + W_PHASE * (pp == AWAY) + W_RESTART * (rp == AWAY)
    E[:, 2] = D_BIAS + W_SETPIECE * dead[lo:hi] + W_OOB * oob[lo:hi]
    return E

def viterbi(E, p_sw, p_in, p_out):
    n = len(E)
    T = np.array([[0, p_sw, p_in], [p_sw, 0, p_in], [p_out, p_out, 0]], np.float32)  # from -> to
    score = E[0].copy()
    back = np.zeros((n, 3), np.int8)
    for i in range(1, n):
        cand = score[:, None] - T
        back[i] = cand.argmax(0)
        score = cand.max(0) + E[i]
    path = np.zeros(n, np.int8)
    path[-1] = score.argmax()
    for i in range(n - 1, 0, -1):
        path[i - 1] = back[i, path[i]]
    return path          # 0=H, 1=A, 2=D

def absorb_short_spells(path, min_len):
    """Merge H/A runs shorter than min_len into the previous state."""
    out = path.copy()
    edges = np.r_[0, np.where(out[1:] != out[:-1])[0] + 1, len(out)]
    for a, b in zip(edges[:-1], edges[1:]):
        if out[a] != 2 and (b - a) < min_len and a > 0:
            out[a:b] = out[a - 1]
    return out

def build_submission_v2(half_blocks, p_sw=P_SWITCH):
    labels = np.full(N_FRAMES, "X", dtype="<U1")
    for _, first, last in half_blocks:
        lo, hi = first, last + 1
        path = viterbi(build_emissions(lo, hi), p_sw, P_ENTER_D, P_EXIT_D)
        path = absorb_short_spells(path, MIN_SPELL)
        labels[lo:hi] = np.array(["H", "A", "D"])[path]
    return labels

def spell_stats(labels, half_blocks):
    n, durs = 0, []
    for _, first, last in half_blocks:
        seg = labels[first:last + 1]
        edges = np.r_[0, np.where(seg[1:] != seg[:-1])[0] + 1, len(seg)]
        for a, b in zip(edges[:-1], edges[1:]):
            if seg[a] in ("H", "A"):
                n += 1; durs.append((b - a) / FPS)
    return n, float(np.mean(durs))

labels2 = build_submission_v2(HALF_BLOCKS)
validate_labels(labels2)
n, md = spell_stats(labels2, HALF_BLOCKS)
print(f"possessions: {n} | mean duration: {md:.1f}s")
out_csv2 = f"{OUT_DIR}/possession_submission_v2.csv"
write_submission(out_csv2, labels2)

validate OK: 199319 rows
A    53787
D    46403
H    58667
X    40462
possessions: 154 | mean duration: 29.2s


'/content/drive/MyDrive/Game Theory Barcelona/submissions/possession_submission_v2.csv'

In [ ]:
# ---- Overrides: fixes the plateau bug ----
P_ENTER_D = 40.0
P_EXIT_D  = 40.0
MIN_SPELL = 25

def build_submission_v2(half_blocks, p_sw=10.0, p_in=P_ENTER_D, p_out=P_EXIT_D):
    labels = np.full(N_FRAMES, "X", dtype="<U1")
    for _, first, last in half_blocks:
        lo, hi = first, last + 1
        path = viterbi(build_emissions(lo, hi), p_sw, p_in, p_out)
        path = absorb_short_spells(path, MIN_SPELL)
        labels[lo:hi] = np.array(["H", "A", "D"])[path]
    return labels

def dead_share(labels, half_blocks):
    tot = dead_n = 0
    for _, a, b in half_blocks:
        seg = labels[a:b + 1]
        tot += len(seg)
        dead_n += (seg == "D").sum()
    return dead_n / tot

for p in (4, 6, 8, 10, 14, 20, 30):
    L = build_submission_v2(HALF_BLOCKS, p_sw=p)
    n, md = spell_stats(L, HALF_BLOCKS)
    print(f"P_SWITCH={p:>3}  possessions={n:>4}  mean={md:5.1f}s  dead={dead_share(L, HALF_BLOCKS):.1%}")

P_SWITCH=  4  possessions= 417  mean= 10.8s  dead=28.9%
P_SWITCH=  6  possessions= 381  mean= 11.9s  dead=28.9%
P_SWITCH=  8  possessions= 340  mean= 13.3s  dead=28.9%
P_SWITCH= 10  possessions= 300  mean= 15.1s  dead=28.9%
P_SWITCH= 14  possessions= 242  mean= 18.7s  dead=28.9%
P_SWITCH= 20  possessions= 198  mean= 22.8s  dead=28.9%
P_SWITCH= 30  possessions= 157  mean= 28.8s  dead=28.9%


In [ ]:
BEST_P = 9.0

labels2 = build_submission_v2(HALF_BLOCKS, p_sw=BEST_P)
validate_labels(labels2)
n, md = spell_stats(labels2, HALF_BLOCKS)
print(f"possessions: {n} | mean duration: {md:.1f}s | dead: {dead_share(labels2, HALF_BLOCKS):.1%}")

out_csv2 = f"{OUT_DIR}/possession_submission_v2.csv"
write_submission(out_csv2, labels2)
print("Saved:", out_csv2)

# Pipeline check on the public 5 minutes (alignment only, don't tune on it)
r = pm.evaluate(pm.label_codes(pm.read_label_csv(
        os.path.join(SCORE_DIR, "BAR-ATH_possession_gt_public.csv"))),
        pm.label_codes(labels2))
print(f"public spellF1 {r['spell_f1_mean']:.4f} | macroF1 {r['macro_f1']:.4f} | preds {r['n_pred_spells']} (GT 17)")

validate OK: 199319 rows
A    49116
D    45832
H    63909
X    40462
possessions: 320 | mean duration: 14.1s | dead: 28.9%
Saved: /content/drive/MyDrive/Game Theory Barcelona/submissions/possession_submission_v2.csv
public spellF1 0.2595 | macroF1 0.6406 | preds 20 (GT 17)


In [ ]:
# 1. Where do the D frames come from? (full match, no ground truth needed)
g = np.zeros(N_FRAMES, bool)
for _, a, b in HALF_BLOCKS:
    g[a:b + 1] = True
d = (labels2 == "D")
print("D frames total      :", d[g].sum(), f"({d[g].mean():.1%} of all graded frames)")
print("  set piece tagged  :", (d & dead & g).sum())
print("  ball out of bounds:", (d & oob & g).sum())
print("  neither (bias)    :", (d & ~dead & ~oob & g).sum())

# 2. Public window: frame-level confusion, rows = truth, cols = mine
gt_lab = pd.read_csv(os.path.join(SCORE_DIR, "BAR-ATH_possession_gt_public.csv"))["label"].values
w = gt_lab != "X"
print("\nPublic window (truth rows, mine cols):")
print(pd.crosstab(pd.Series(gt_lab[w], name="truth"), pd.Series(labels2[w], name="mine")))

D frames total      : 45832 (28.9% of all graded frames)
  set piece tagged  : 45124
  ball out of bounds: 2749
  neither (bias)    : 186

Public window (truth rows, mine cols):
mine      A    D     H
truth                 
A      1632  226   719
D       251  692   526
H       417  296  2741


In [ ]:
# 1. Set-piece length by type (full match, no ground truth)
rows = [(ev["tags"].get("Type", "?"), (ev["end_frame"] - ev["start_frame"]) / FPS)
        for ev in events if ev["code"].upper() == "SET PIECES"]
print(pd.DataFrame(rows, columns=["type", "secs"]).groupby("type")["secs"]
        .agg(["count", "mean", "min", "max"]).round(1))

# 2. Public window: dead-ball stretches, truth vs mine (structure only)
def runs(lab, lo, hi):
    out, s = [], lo
    for i in range(lo + 1, hi + 2):
        if i == hi + 1 or lab[i] != lab[s]:
            if lab[s] == "D":
                out.append((s, i - 1, round((i - s) / FPS, 1)))
            s = i
    return out

print("\nTRUTH D stretches (start, end, secs):", runs(gt_lab, 11185, 18684))
print("MINE  D stretches (start, end, secs):", runs(labels2, 11185, 18684))

                       count  mean   min   max
type                                          
Corner Kick               12  35.0  35.0  35.0
Free Kick                  5  35.0  35.0  35.0
Goal Kick                 10  35.0  35.0  35.0
Kick Off (After Goal)      4   5.0   5.0   5.0
Kick Off (Start)           2   5.0   5.0   5.0
Throw-in                  26  35.0  35.0  35.0

TRUTH D stretches (start, end, secs): [(12851, 12988, 5.5), (15889, 16399, 20.4), (17380, 17837, 18.3), (18027, 18388, 14.5)]
MINE  D stretches (start, end, secs): [(11185, 11221, 1.5), (17508, 18684, 47.1)]


In [ ]:
L_, W_ = PITCH
bxm, bym = bx * L_, by * W_
spd = np.full(N_FRAMES, np.nan)
both = np.isfinite(bxm[1:]) & np.isfinite(bxm[:-1])
spd[1:][both] = np.hypot(np.diff(bxm)[both], np.diff(bym)[both]) * FPS

def summ(mask, name):
    sp = spd[mask]; sp = sp[np.isfinite(sp)]
    still = (sp < 1).mean() if len(sp) else float("nan")
    med = np.median(sp) if len(sp) else float("nan")
    print(f"{name:26s} frames {mask.sum():>5} | ball seen {np.isfinite(bx[mask]).mean():4.0%} | "
          f"outside pitch {oob[mask].mean():4.0%} | median speed {med:5.1f} m/s | still (<1 m/s) {still:4.0%}")

truth_dead = (gt_lab == "D")
truth_live = (gt_lab == "H") | (gt_lab == "A")
summ(truth_dead, "TRUE dead frames")
summ(truth_live, "TRUE live frames")
for a, b in [(12851, 12988), (15889, 16399), (17380, 17837), (18027, 18388)]:
    m = np.zeros(N_FRAMES, bool); m[a:b + 1] = True
    summ(m, f"dead stretch {a}-{b}")

print("\nSet-piece tags overlapping the public window:")
for ev in events:
    if ev["code"].upper() == "SET PIECES" and ev["end_frame"] > 11185 and ev["start_frame"] < 18684:
        print(ev["tags"].get("Type"), "|", ev["tags"].get("Team"), "|", ev["start_frame"], "-", ev["end_frame"])

TRUE dead frames           frames  1469 | ball seen  66% | outside pitch   7% | median speed   2.7 m/s | still (<1 m/s)  10%
TRUE live frames           frames  6031 | ball seen  94% | outside pitch   0% | median speed   5.7 m/s | still (<1 m/s)   7%
dead stretch 12851-12988   frames   138 | ball seen   7% | outside pitch   0% | median speed  10.1 m/s | still (<1 m/s)   0%
dead stretch 15889-16399   frames   511 | ball seen 100% | outside pitch   0% | median speed   2.8 m/s | still (<1 m/s)   7%
dead stretch 17380-17837   frames   458 | ball seen  20% | outside pitch   1% | median speed  10.9 m/s | still (<1 m/s)   0%
dead stretch 18027-18388   frames   362 | ball seen 100% | outside pitch  27% | median speed   1.7 m/s | still (<1 m/s)  18%

Set-piece tags overlapping the public window:
Kick Off (Start) | N/A | 11097 - 11222
Goal Kick | Barcelona | 18320 - 19195
Free Kick | Atletic Club | 17533 - 18408


In [ ]:
# ---- v3: tags say WHERE to look, the ball decides IF it is dead ----
PRE_ZONE = 250    # 10 s before tag start (tag starts are fuzzy)
W_ZONE   = 1.5    # dead evidence: inside a tag zone AND ball missing / outside pitch

zone = np.zeros(N_FRAMES, bool)
for ev in events:
    if ev["code"].upper() != "SET PIECES":
        continue
    zone[max(0, ev["start_frame"] - PRE_ZONE): min(N_FRAMES, ev["end_frame"])] = True

missing  = ~np.isfinite(bx)
ball_off = missing | oob

def build_emissions(lo, hi):
    s = ctrl[lo:hi]; n = hi - lo
    E = np.zeros((n, 3), np.float32)
    pp = phase_prior[lo:hi]; rp = restart_prior[lo:hi]
    E[:, 0] = W_BALL * s + W_PHASE * (pp == HOME) + W_RESTART * (rp == HOME)
    E[:, 1] = -W_BALL * s + W_PHASE * (pp == AWAY) + W_RESTART * (rp == AWAY)
    E[:, 2] = D_BIAS + W_ZONE * (zone[lo:hi] & ball_off[lo:hi]) + W_OOB * oob[lo:hi]
    return E

def d_stretches(labels):
    out = []
    for _, a, b in HALF_BLOCKS:
        seg = labels[a:b + 1] == "D"
        edges = np.r_[0, np.where(seg[1:] != seg[:-1])[0] + 1, len(seg)]
        out += [(y - x) / FPS for x, y in zip(edges[:-1], edges[1:]) if seg[x]]
    return np.array(out)

labels3 = build_submission_v2(HALF_BLOCKS, p_sw=9.0)
validate_labels(labels3)

# Structural comparison on the full match (no ground truth involved)
for name, L in (("v2", labels2), ("v3", labels3)):
    ds = d_stretches(L); n, md = spell_stats(L, HALF_BLOCKS)
    print(f"{name}: possessions={n}  mean={md:.1f}s | dead={dead_share(L, HALF_BLOCKS):.1%} "
          f"| dead stretches={len(ds)}  median={np.median(ds):.1f}s  >40s: {(ds > 40).mean():.0%}")

write_submission(f"{OUT_DIR}/possession_submission_v3.csv", labels3)

# Public window: alignment check and confusion only
r = pm.evaluate(pm.label_codes(pm.read_label_csv(
        os.path.join(SCORE_DIR, "BAR-ATH_possession_gt_public.csv"))), pm.label_codes(labels3))
print(f"\npublic spellF1 {r['spell_f1_mean']:.4f} | macroF1 {r['macro_f1']:.4f} | preds {r['n_pred_spells']} (GT 17)")
print(pd.crosstab(pd.Series(gt_lab[w], name="truth"), pd.Series(labels3[w], name="mine")))

validate OK: 199319 rows
A    56782
D    26211
H    75864
X    40462
v2: possessions=320  mean=14.1s | dead=28.9% | dead stretches=50  median=36.0s  >40s: 18%
v3: possessions=436  mean=12.2s | dead=16.5% | dead stretches=78  median=10.0s  >40s: 4%

public spellF1 0.3487 | macroF1 0.6644 | preds 22 (GT 17)
mine      A    D     H
truth                 
A      1910    1   666
D       212  467   790
H       417    0  3037


In [ ]:
# ---- v4: add a "ball is slow" dead cue, only inside tag zones ----
sm_spd = pd.Series(spd).rolling(50, center=True, min_periods=10).median().values

graded = np.zeros(N_FRAMES, bool)
for _, a, b in HALF_BLOCKS:
    graded[a:b + 1] = True

ref = sm_spd[graded & ~zone & np.isfinite(sm_spd)]
SLOW_THR = float(np.percentile(ref, 20))
print(f"slow-ball threshold (20th pct of speed outside zones): {SLOW_THR:.2f} m/s")

W_SLOW = 1.5
slow = zone & np.isfinite(sm_spd) & (sm_spd < SLOW_THR)

def build_emissions(lo, hi):
    s = ctrl[lo:hi]; n = hi - lo
    E = np.zeros((n, 3), np.float32)
    pp = phase_prior[lo:hi]; rp = restart_prior[lo:hi]
    E[:, 0] = W_BALL * s + W_PHASE * (pp == HOME) + W_RESTART * (rp == HOME)
    E[:, 1] = -W_BALL * s + W_PHASE * (pp == AWAY) + W_RESTART * (rp == AWAY)
    E[:, 2] = (D_BIAS + W_ZONE * (zone[lo:hi] & ball_off[lo:hi])
               + W_OOB * oob[lo:hi] + W_SLOW * slow[lo:hi])
    return E

print(f"slow-ball frames: {(slow & graded).sum()} of {graded.sum()} graded")

# Structure for several switch costs (no ground truth involved)
for p in (6, 8, 10, 12, 14):
    L = build_submission_v2(HALF_BLOCKS, p_sw=p)
    ds = d_stretches(L); n, md = spell_stats(L, HALF_BLOCKS)
    print(f"P_SWITCH={p:>3} possessions={n:>4} mean={md:5.1f}s | dead={dead_share(L, HALF_BLOCKS):.1%} "
          f"| dead stretches={len(ds)} median={np.median(ds):.1f}s  >40s: {(ds > 40).mean():.0%}")

# Public window: alignment and confusion only
labels4 = build_submission_v2(HALF_BLOCKS, p_sw=9.0)
r = pm.evaluate(pm.label_codes(pm.read_label_csv(
        os.path.join(SCORE_DIR, "BAR-ATH_possession_gt_public.csv"))), pm.label_codes(labels4))
print(f"\nP_SWITCH=9 public spellF1 {r['spell_f1_mean']:.4f} | macroF1 {r['macro_f1']:.4f} | preds {r['n_pred_spells']} (GT 17)")
print(pd.crosstab(pd.Series(gt_lab[w], name="truth"), pd.Series(labels4[w], name="mine")))

slow-ball threshold (20th pct of speed outside zones): 2.37 m/s
slow-ball frames: 7339 of 158857 graded
P_SWITCH=  6 possessions= 500 mean= 10.2s | dead=19.5% | dead stretches=79 median=13.5s  >40s: 5%
P_SWITCH=  8 possessions= 443 mean= 11.5s | dead=19.6% | dead stretches=79 median=13.5s  >40s: 5%
P_SWITCH= 10 possessions= 400 mean= 12.8s | dead=19.6% | dead stretches=79 median=13.5s  >40s: 5%
P_SWITCH= 12 possessions= 353 mean= 14.4s | dead=19.8% | dead stretches=79 median=13.5s  >40s: 5%
P_SWITCH= 14 possessions= 320 mean= 15.9s | dead=19.9% | dead stretches=79 median=13.5s  >40s: 5%

P_SWITCH=9 public spellF1 0.3692 | macroF1 0.7262 | preds 22 (GT 17)
mine      A    D     H
truth                 
A      1910    1   666
D       212  699   558
H       417    0  3037


In [ ]:
BEST_P = 12.0

labels_final = build_submission_v2(HALF_BLOCKS, p_sw=BEST_P)
validate_labels(labels_final)
n, md = spell_stats(labels_final, HALF_BLOCKS)
print(f"possessions: {n} | mean duration: {md:.1f}s | dead: {dead_share(labels_final, HALF_BLOCKS):.1%}")

out_final = f"{OUT_DIR}/possession_submission_v4.csv"
write_submission(out_final, labels_final)
print("Saved:", out_final)

r = pm.evaluate(pm.label_codes(pm.read_label_csv(
        os.path.join(SCORE_DIR, "BAR-ATH_possession_gt_public.csv"))), pm.label_codes(labels_final))
print(f"public spellF1 {r['spell_f1_mean']:.4f} | macroF1 {r['macro_f1']:.4f} | preds {r['n_pred_spells']} (GT 17)")

validate OK: 199319 rows
A    54793
D    31528
H    72536
X    40462
possessions: 353 | mean duration: 14.4s | dead: 19.8%
Saved: /content/drive/MyDrive/Game Theory Barcelona/submissions/possession_submission_v4.csv
public spellF1 0.4000 | macroF1 0.7299 | preds 20 (GT 17)


# Idea 3: touch-event chains ("count the kicks, not the frames")

**Labels:** **H** = Barcelona has the ball, **A** = Athletic has the ball, **D** = dead ball.

## The idea in one line
Ideas 1 and 2 asked *"who is near the ball?"* at **every frame**. Idea 3 asks only at the **moments the ball is actually touched**, then reads possession from the chain of touches.

## How it works
1. **Find touches.** When a player passes, shoots or controls the ball, the ball suddenly changes speed or direction. We compare the ball's velocity over the 0.2 s before each frame with the 0.2 s after. A sharp peak above `TAU` m/s is a touch.
2. **Who touched it?** The nearest player at that moment. It only counts if that player is within `RADIUS` metres and clearly closer than the nearest opponent (a 0.5 m margin). Contested or far-away peaks are ignored.
3. **Build the chain.** Touches in order, for example `H H H A A H`. A possession is a run of the same team, and it starts at the first touch of the new team. Between two touches the ball is in the air and still belongs to the passer's team.
4. **Ignore deflections.** A single touch by the other team between two touches of the first team within 3 seconds (`H A H`) is a block or deflection, so the possession stays with the first team.
5. **Dead ball.** We reuse the dead-ball detector from idea 2. After a stoppage, the team is set by the first touch.
6. **Short gaps.** A missing ball for 0.5 s or less is filled in by a straight line. Longer gaps keep the last team.

## Example
Touch times (s): `2.0 H`, `5.5 H`, `9.0 H`, `12.5 A`, `15.0 A`, `16.2 H`, `20.0 H`.

- The `H H H` run is one Barcelona possession, from 2.0 s until 12.5 s.
- The `A A` run is one Athletic possession, from 12.5 s until 16.2 s.
- A new Barcelona possession starts at 16.2 s.

**Deflection case:** `H` at 10 s, `A` at 11 s, `H` at 12 s. The lone `A` touch is wiped out and the possession stays **H**.

## Why it is different
| | Idea 2 | Idea 3 |
|---|---|---|
| Unit | every frame | discrete touches |
| Evidence | "who is near" at all times | "who hit the ball" only at kicks |
| Switching rule | a price paid per frame | a new team must touch the ball |
| Boundary position | where the evidence flips | at the touch itself |

## Main knobs
- `TAU` (touch threshold in m/s): higher gives fewer touches and fewer possessions.
- `RADIUS`: how close the player must be to count as the toucher.

## Weak spots
- Tracking noise can create fake touches. The player-distance check removes most of them.
- A missing ball means missed touches, so the chain holds the last team longer.
- Dead ball is only as good as idea 2's detector.

In [ ]:
# ===== Idea 3: touch-event chains =====
def interp_short_gaps(a, gap_max):
    out = np.asarray(a, dtype=np.float64).copy()
    isn = np.isnan(out)
    if not isn.any():
        return out
    edges = np.flatnonzero(np.diff(np.r_[0, isn.astype(np.int8), 0]))
    for s, e in zip(edges[::2], edges[1::2]):          # NaN run is [s, e)
        if s == 0 or e == len(out):
            continue
        if e - s <= gap_max:
            out[s:e] = np.interp(np.arange(s, e), [s - 1, e], [out[s - 1], out[e]])
    return out

def shift(a, k):                                        # returns a[t+k]
    out = np.full(len(a), np.nan)
    if k > 0:   out[:-k] = a[k:]
    elif k < 0: out[-k:] = a[:k]
    else:       out[:] = a
    return out

def clean_touches(tf, tt, flank_gap, passes=5):
    """Remove a lone touch by one team sandwiched between touches of the other team (H A H)."""
    tf = [int(x) for x in tf]; tt = [int(x) for x in tt]
    for _ in range(passes):
        runs, i = [], 0
        while i < len(tt):
            j = i
            while j + 1 < len(tt) and tt[j + 1] == tt[i]:
                j += 1
            runs.append((tt[i], i, j)); i = j + 1
        flipped, r = False, 1
        while r < len(runs) - 1:
            team, i0, i1 = runs[r]
            if i0 == i1 and runs[r - 1][0] == runs[r + 1][0] \
               and tf[runs[r + 1][1]] - tf[i0] < flank_gap:
                tt[i0] = runs[r - 1][0]; flipped = True; r += 2
            else:
                r += 1
        if not flipped:
            break
    return np.array(tf, dtype=np.int64), np.array(tt, dtype=np.int8)

# ---- Precompute once (heavy part) ----
GAP_MAX, K, PEAK_HALF = 12, 5, 8          # 0.5 s gap fill | 0.2 s velocity window | 0.32 s min touch spacing
L_, W_ = PITCH
pxm = interp_short_gaps(bx.astype(np.float64) * L_, GAP_MAX)
pym = interp_short_gaps(by.astype(np.float64) * W_, GAP_MAX)
dHi = interp_short_gaps(dH, GAP_MAX)
dAi = interp_short_gaps(dA, GAP_MAX)

vbx = (pxm - shift(pxm, -K)) * FPS / K;  vby = (pym - shift(pym, -K)) * FPS / K   # velocity before
vax = (shift(pxm, K) - pxm) * FPS / K;   vay = (shift(pym, K) - pym) * FPS / K    # velocity after
dv = np.nan_to_num(np.hypot(vax - vbx, vay - vby), nan=0.0)                       # velocity change (m/s)
rollmax = pd.Series(dv).rolling(2 * PEAK_HALF + 1, center=True, min_periods=1).max().values
near_h = pd.Series(dHi).rolling(11, center=True, min_periods=1).min().values
near_a = pd.Series(dAi).rolling(11, center=True, min_periods=1).min().values

graded_mask = np.zeros(N_FRAMES, bool)
for _, a, b in HALF_BLOCKS:
    graded_mask[a:b + 1] = True

v4_codes = np.array([{"H": 0, "A": 1}.get(c, 2) for c in labels_final], dtype=np.int8)
dead_mask = (labels_final == "D")          # dead-ball detector reused from idea 2

def absorb(path, min_len):
    out = path.copy()
    edges = np.r_[0, np.where(out[1:] != out[:-1])[0] + 1, len(out)]
    for a, b in zip(edges[:-1], edges[1:]):
        if out[a] != 2 and (b - a) < min_len and a > 0:
            out[a:b] = out[a - 1]
    return out

def build_touch_labels(tau=3.0, radius=2.5, margin=0.5, flank_gap=75, min_spell=25, backfill=250):
    idx = np.flatnonzero((dv == rollmax) & (dv >= tau) & graded_mask)      # candidate touches
    hh = np.nan_to_num(near_h[idx], nan=99.0)
    aa = np.nan_to_num(near_a[idx], nan=99.0)
    ok = (np.minimum(hh, aa) <= radius) & (np.abs(hh - aa) >= margin)      # a clear toucher
    tf = idx[ok]; tt = np.where(hh[ok] < aa[ok], 0, 1)
    n_raw = len(idx)
    if len(tf):
        tf, tt = clean_touches(tf, tt, flank_gap)

    path = np.full(N_FRAMES, 2, np.int8)
    for _, first, last in HALF_BLOCKS:
        lo, hi = first, last + 1
        m = (tf >= lo) & (tf < hi)
        tfh, tth = tf[m], tt[m]
        if len(tfh) == 0:                                   # safety: fall back to idea 2 for this half
            path[lo:hi] = v4_codes[lo:hi]; continue
        frames = np.arange(lo, hi)
        j = np.searchsorted(tfh, frames, side="right") - 1
        team = tth[np.clip(j, 0, None)].copy()              # hold the last toucher's team
        live = ~dead_mask[lo:hi]
        edges = np.flatnonzero(np.diff(np.r_[0, live.astype(np.int8), 0]))
        for s, e in zip(edges[::2], edges[1::2]):           # after a stoppage, the first toucher owns the restart
            k = np.searchsorted(tfh, lo + s)
            if k < len(tfh) and tfh[k] < lo + e and tfh[k] - (lo + s) <= backfill:
                team[s: tfh[k] - lo] = tth[k]
        team[~live] = 2
        path[lo:hi] = absorb(team.astype(np.int8), min_spell)

    labels = np.full(N_FRAMES, "X", dtype="<U1")
    for _, first, last in HALF_BLOCKS:
        labels[first:last + 1] = np.array(["H", "A", "D"])[path[first:last + 1]]
    return labels, int(n_raw), int(len(tf))

print("Idea 3 builder ready.")

Idea 3 builder ready.


In [ ]:
def agree(a, b):
    m = np.isin(a, ["H", "A"]) & np.isin(b, ["H", "A"])
    return (a[m] == b[m]).mean()

live_min = (graded_mask & ~dead_mask).sum() / FPS / 60
print(f"live graded minutes (from idea 2's dead mask): {live_min:.1f}\n")

for radius in (2.0, 3.0):
    for tau in (2.0, 3.0, 4.0, 5.0):
        L3, n_raw, n_kept = build_touch_labels(tau=tau, radius=radius)
        n, md = spell_stats(L3, HALF_BLOCKS)
        print(f"R={radius} TAU={tau}: peaks={n_raw:>5} kept={n_kept:>5} ({n_kept / live_min:4.1f}/live min) "
              f"| possessions={n:>4} mean={md:5.1f}s | agree with idea 2: {agree(L3, labels_final):.1%}")

live graded minutes (from idea 2's dead mask): 84.9

R=2.0 TAU=2.0: peaks= 1537 kept= 1085 (12.8/live min) | possessions= 219 mean= 23.3s | agree with idea 2: 66.3%
R=2.0 TAU=3.0: peaks= 1010 kept=  700 ( 8.2/live min) | possessions= 190 mean= 26.8s | agree with idea 2: 63.4%
R=2.0 TAU=4.0: peaks=  558 kept=  342 ( 4.0/live min) | possessions= 157 mean= 32.4s | agree with idea 2: 60.7%
R=2.0 TAU=5.0: peaks=  289 kept=  146 ( 1.7/live min) | possessions= 118 mean= 43.2s | agree with idea 2: 51.2%
R=3.0 TAU=2.0: peaks= 1537 kept= 1202 (14.2/live min) | possessions= 223 mean= 22.8s | agree with idea 2: 68.0%
R=3.0 TAU=3.0: peaks= 1010 kept=  787 ( 9.3/live min) | possessions= 194 mean= 26.3s | agree with idea 2: 64.5%
R=3.0 TAU=4.0: peaks=  558 kept=  404 ( 4.8/live min) | possessions= 176 mean= 28.9s | agree with idea 2: 61.2%
R=3.0 TAU=5.0: peaks=  289 kept=  188 ( 2.2/live min) | possessions= 126 mean= 40.4s | agree with idea 2: 49.9%


In [ ]:
BEST_TAU, BEST_R = 3.0, 2.5

labels5, n_raw, n_kept = build_touch_labels(tau=BEST_TAU, radius=BEST_R)
validate_labels(labels5)
n, md = spell_stats(labels5, HALF_BLOCKS)
print(f"touches kept: {n_kept} | possessions: {n} | mean: {md:.1f}s | dead: {dead_share(labels5, HALF_BLOCKS):.1%}")

out5 = f"{OUT_DIR}/possession_submission_v5_touch.csv"
write_submission(out5, labels5)
print("Saved:", out5)

# Public window: pipeline check only. Compare v4 and v5 side by side.
gt_pub = pm.label_codes(pm.read_label_csv(os.path.join(SCORE_DIR, "BAR-ATH_possession_gt_public.csv")))
for name, L in (("v4 (idea 2)", labels_final), ("v5 (idea 3)", labels5)):
    r = pm.evaluate(gt_pub, pm.label_codes(L))
    print(f"{name}: spellF1 {r['spell_f1_mean']:.4f} | macroF1 {r['macro_f1']:.4f} | preds {r['n_pred_spells']} (GT 17)")

validate OK: 199319 rows
A    59098
D    31528
H    68231
X    40462
touches kept: 769 | possessions: 197 | mean: 25.9s | dead: 19.8%
Saved: /content/drive/MyDrive/Game Theory Barcelona/submissions/possession_submission_v5_touch.csv
v4 (idea 2): spellF1 0.4000 | macroF1 0.7299 | preds 20 (GT 17)
v5 (idea 3): spellF1 0.5231 | macroF1 0.7405 | preds 9 (GT 17)


# Idea 3 retuned on external data (final method)

**Labels:** **H** = Barcelona has the ball, **A** = Athletic, **D** = dead ball.

**What changed:** the touch-chain method (idea 3) is now the main engine, and every parameter was tuned on *external* data only: Metrica Sports' public sample games 1 and 2 (https://github.com/metrica-sports/sample-data). Labels for those games are built automatically from their event files. Nothing was tuned on the Barcelona ground truth or video.

**How it works**
1. Find touches: sudden changes in ball velocity. The threshold is set by a rate, the strongest 24 peaks per graded minute, so it adapts to how smooth the tracking is.
2. Give each touch to the nearest player's team (within 3 m; a tie goes unassigned).
3. Possession = the chain of touching teams. A lone opposing touch between two touches of the same team within 1 s is a deflection and is ignored.
4. Dead ball = ball missing or outside the pitch for 3 s or more (gaps up to 2 s are closed, runs under 2 s dropped).
5. After a stoppage, the first touch decides who restarts.

**External results (spellF1, 2 games):** nearest+vote 0.55 / 0.54, decoder 0.61 / 0.65, touch chains 0.79 / 0.82.

**Removed because they were first set while looking at the public ground truth:** Smart Tagging zones, the 10 s pre-zone, the ball-missing zone gate and the slow-ball cue.

**Limits:** external labels are a proxy built from events; the external tracking provider differs; the real match's dead-ball behaviour may differ.

In [ ]:
# ===== External tuning data: Metrica Sports sample games 1 and 2 (public, anonymised) =====
# Source: https://github.com/metrica-sports/sample-data  (README asks: be responsible, acknowledge the source)
import urllib.request, os
EXT_DIR = "/content/metrica_ext"
os.makedirs(EXT_DIR, exist_ok=True)
BASE_EXT = "https://raw.githubusercontent.com/metrica-sports/sample-data/master/data"
for g in (1, 2):
    for f in ("RawEventsData", "RawTrackingData_Home_Team", "RawTrackingData_Away_Team"):
        dest = f"{EXT_DIR}/G{g}_{f}.csv"
        if not os.path.isfile(dest):
            print("downloading", g, f)
            urllib.request.urlretrieve(f"{BASE_EXT}/Sample_Game_{g}/Sample_Game_{g}_{f}.csv", dest)

def load_tracking_ext(game):
    H = pd.read_csv(f"{EXT_DIR}/G{game}_RawTrackingData_Home_Team.csv", header=None, skiprows=3, low_memory=False)
    A = pd.read_csv(f"{EXT_DIR}/G{game}_RawTrackingData_Away_Team.csv", header=None, skiprows=3, low_memory=False)
    n = min(len(H), len(A)); H, A = H.iloc[:n], A.iloc[:n]
    hp = H.iloc[:, 3:-2].values.astype(float).reshape(n, -1, 2)
    ap = A.iloc[:, 3:-2].values.astype(float).reshape(n, -1, 2)
    ball = H.iloc[:, -2:].values.astype(float)
    return hp, ap, ball

def nearest_dists_ext(hp, ap, ball, pitch=(105.0, 68.0)):
    L, W = pitch
    b = ball * np.array([L, W])
    def nd(p):
        d = np.hypot(p[:, :, 0] * L - b[:, None, 0], p[:, :, 1] * W - b[:, None, 1])
        out = np.full(len(d), np.nan); ok = ~np.all(np.isnan(d), axis=1)
        out[ok] = np.nanmin(d[ok], axis=1); return out
    return nd(hp).astype(np.float32), nd(ap).astype(np.float32)

def event_labels_ext(ev, n_frames):
    """Automatic possession labels from the event stream (external data, no hand labelling)."""
    ev = ev.copy(); ev["sf"] = ev["Start Frame"].astype(int); ev["ef"] = ev["End Frame"].astype(int)
    ev = ev.sort_values("sf", kind="stable").reset_index(drop=True)
    tmap = {"Home": 0, "Away": 1}; lab = np.full(n_frames, -1, np.int8); pts = []
    for r in ev.itertuples():
        st = str(r.Subtype) if isinstance(r.Subtype, str) else ""
        has = r.Type in ("PASS", "SHOT", "SET PIECE", "RECOVERY") or \
              (r.Type == "CHALLENGE" and st.endswith("WON") and "FAULT" not in st and "ADVANTAGE" not in st)
        if has: pts.append((r.sf, tmap[r.Team]))
    pts = sorted(pts)
    for (f0, t0), nxt in zip(pts, pts[1:] + [(n_frames + 1, 0)]):
        lab[max(f0 - 1, 0): min(nxt[0] - 1, n_frames)] = t0
    stops = []
    for r in ev.itertuples():
        st = str(r.Subtype) if isinstance(r.Subtype, str) else ""
        if r.Type == "BALL OUT": stops.append(r.ef)
        elif r.Type == "FAULT RECEIVED": stops.append(r.sf)
        elif r.Type == "BALL LOST" and st in ("OFFSIDE", "HAND BALL", "REFEREE HIT"): stops.append(r.ef)
        elif r.Type == "SHOT" and "GOAL" in st: stops.append(r.ef)
    stops = np.array(sorted(stops)); dead = np.zeros(n_frames, bool); prev = 0
    for rs in ev[ev.Type == "SET PIECE"].sf.values:
        cand = stops[(stops > prev) & (stops < rs)]
        if len(cand) and rs - cand[0] < 25 * 180: dead[cand[0] - 1: rs - 1] = True
        prev = rs
    out = np.full(n_frames, "X", dtype="<U1")
    for p, g in ev.groupby("Period"):
        a, b = int(g.sf.min()), int(g.ef.max()); seg = slice(a - 1, min(b, n_frames))
        s = np.where(lab[seg] == 0, "H", np.where(lab[seg] == 1, "A", "X")); s[dead[seg]] = "D"; out[seg] = s
    return out

EXT = {}
for g in (1, 2):
    hp, ap, ball = load_tracking_ext(g)
    ev = pd.read_csv(f"{EXT_DIR}/G{g}_RawEventsData.csv")
    gt_ext = event_labels_ext(ev, len(ball))
    dHe, dAe = nearest_dists_ext(hp, ap, ball)
    EXT[g] = dict(gt=gt_ext, graded=(gt_ext != "X"), bx=ball[:, 0], by=ball[:, 1], dH=dHe, dA=dAe)
    print(f"game {g}: {EXT[g]['graded'].sum()} graded frames | dead share {np.mean(gt_ext[EXT[g]['graded']] == 'D'):.1%}")

downloading 1 RawEventsData
downloading 1 RawTrackingData_Home_Team
downloading 1 RawTrackingData_Away_Team
downloading 2 RawEventsData
downloading 2 RawTrackingData_Home_Team
downloading 2 RawTrackingData_Away_Team
game 1: 142074 graded frames | dead share 37.8%
game 2: 140114 graded frames | dead share 34.7%


In [ ]:
# ===== Engines + dead-ball detector (shared by the external tuning and by the real match) =====
def periods(graded):
    e = np.flatnonzero(np.diff(np.r_[0, graded.astype(np.int8), 0]))
    return list(zip(e[::2], e[1::2]))                 # [lo, hi)

def run_mask(mask, min_len):                          # keep True-runs of at least min_len frames
    out = np.zeros(len(mask), bool)
    e = np.flatnonzero(np.diff(np.r_[0, mask.astype(np.int8), 0]))
    for s, t in zip(e[::2], e[1::2]):
        if t - s >= min_len: out[s:t] = True
    return out

def close_gaps(mask, max_gap):                        # fill False-gaps up to max_gap between True runs
    out = mask.copy(); e = np.flatnonzero(np.diff(np.r_[0, (~mask).astype(np.int8), 0]))
    for s, t in zip(e[::2], e[1::2]):
        if s > 0 and t < len(mask) and t - s <= max_gap: out[s:t] = True
    return out

class DeadDetector:
    """Ball-only dead-ball detector: ball missing or outside the pitch for long enough (optional: nearly still)."""
    def __init__(self, bx, by, fps=25, pitch=(105.0, 68.0), eps=0.003):
        L, W = pitch
        self.missing = ~np.isfinite(bx); s = np.isfinite(bx); self.oob = np.zeros(len(bx), bool)
        self.oob[s] = (bx[s] < -eps) | (bx[s] > 1 + eps) | (by[s] < -eps) | (by[s] > 1 + eps)
        px, py = bx.astype(np.float64) * L, by.astype(np.float64) * W
        sp = np.full(len(bx), np.nan); ok = np.isfinite(px[1:]) & np.isfinite(px[:-1])
        sp[1:][ok] = np.hypot(np.diff(px)[ok], np.diff(py)[ok]) * fps
        self.speed = pd.Series(sp).rolling(50, center=True, min_periods=10).median().values
    def mask(self, miss_min=75, slow_thr=0.0, slow_min=75, close=50, final_min=50, oob_min=10):
        d = run_mask(self.missing | self.oob, miss_min) | run_mask(self.oob, oob_min)
        if slow_thr > 0:
            d |= run_mask(np.nan_to_num(self.speed, nan=99.0) < slow_thr, slow_min)
        return run_mask(close_gaps(d, close), final_min)

class TouchEngine:
    """Idea 3: touches from sudden ball-velocity changes; possession = chain of touching teams."""
    def __init__(self, bx, by, dH, dA, graded, fps=25, pitch=(105.0, 68.0), gap_max=12, K=5, peak_half=8):
        L, W = pitch
        pxm = interp_short_gaps(bx.astype(np.float64) * L, gap_max)
        pym = interp_short_gaps(by.astype(np.float64) * W, gap_max)
        dHi = interp_short_gaps(dH, gap_max); dAi = interp_short_gaps(dA, gap_max)
        vbx = (pxm - shift(pxm, -K)) * fps / K; vby = (pym - shift(pym, -K)) * fps / K
        vax = (shift(pxm, K) - pxm) * fps / K;  vay = (shift(pym, K) - pym) * fps / K
        self.dv = np.nan_to_num(np.hypot(vax - vbx, vay - vby), nan=0.0)
        self.rollmax = pd.Series(self.dv).rolling(2 * peak_half + 1, center=True, min_periods=1).max().values
        self.near_h = pd.Series(dHi).rolling(11, center=True, min_periods=1).min().values
        self.near_a = pd.Series(dAi).rolling(11, center=True, min_periods=1).min().values
        self.graded = graded; self.n = len(bx); self.fps = fps
    def tau_for_rate(self, rate_per_min):
        """Scale-free touch threshold: keep the strongest `rate_per_min` velocity peaks per graded minute."""
        cand = (self.dv == self.rollmax) & (self.dv > 0) & self.graded
        vals = np.sort(self.dv[cand])[::-1]
        n_target = int(rate_per_min * self.graded.sum() / self.fps / 60)
        return float(vals[min(n_target, len(vals)) - 1])
    def labels(self, dead_mask, tau, radius=3.0, margin=0.1, flank_gap=25, min_spell=25, backfill=250):
        idx = np.flatnonzero((self.dv == self.rollmax) & (self.dv >= tau) & self.graded)
        hh = np.nan_to_num(self.near_h[idx], nan=99.0); aa = np.nan_to_num(self.near_a[idx], nan=99.0)
        ok = (np.minimum(hh, aa) <= radius) & (np.abs(hh - aa) >= margin)
        tf = idx[ok]; tt = np.where(hh[ok] < aa[ok], 0, 1)
        if len(tf): tf, tt = clean_touches(tf, tt, flank_gap)
        path = np.full(self.n, 2, np.int8)
        for lo, hi in periods(self.graded):
            m = (tf >= lo) & (tf < hi); tfh, tth = tf[m], tt[m]
            if len(tfh) == 0: continue
            j = np.searchsorted(tfh, np.arange(lo, hi), side="right") - 1
            team = tth[np.clip(j, 0, None)].copy(); live = ~dead_mask[lo:hi]
            e = np.flatnonzero(np.diff(np.r_[0, live.astype(np.int8), 0]))
            for s, t in zip(e[::2], e[1::2]):
                k = np.searchsorted(tfh, lo + s)
                if k < len(tfh) and tfh[k] < lo + t and tfh[k] - (lo + s) <= backfill:
                    team[s: tfh[k] - lo] = tth[k]
            team[~live] = 2
            path[lo:hi] = absorb(team.astype(np.int8), min_spell)
        labels = np.full(self.n, "X", dtype="<U1")
        for lo, hi in periods(self.graded):
            labels[lo:hi] = np.array(["H", "A", "D"])[path[lo:hi]]
        return labels

In [ ]:
# ===== Apply the externally tuned idea 3 to the Barcelona match (tracking only, Smart Tagging not needed) =====
FINAL = dict(rate=24, margin=0.1, flank=25, radius=3.0)                     # chosen on external data
DEAD  = dict(miss_min=75, slow_thr=0.0, slow_min=100, close=50, final_min=50)

dd_real   = DeadDetector(bx, by, pitch=PITCH)
dead_real = dd_real.mask(**DEAD) & graded_mask
te_real   = TouchEngine(bx, by, dH, dA, graded_mask, pitch=PITCH)
tau_real  = te_real.tau_for_rate(FINAL["rate"])
labels6   = te_real.labels(dead_real, tau=tau_real, radius=FINAL["radius"],
                           margin=FINAL["margin"], flank_gap=FINAL["flank"])
validate_labels(labels6)

n, md = spell_stats(labels6, HALF_BLOCKS)
gm = graded_mask.sum() / FPS / 60
print(f"touch threshold tau (from {FINAL['rate']} peaks per graded minute): {tau_real:.2f} m/s")
print(f"possessions: {n} | mean duration: {md:.1f}s | dead share: {dead_share(labels6, HALF_BLOCKS):.1%}")
print("External reference (Metrica games): ~300 possessions, mean 11-13 s, dead share 35-38%")

out6 = f"{OUT_DIR}/possession_submission_v6_touch_external.csv"
write_submission(out6, labels6)
print("Saved:", out6)

validate OK: 199319 rows
A    37109
D    68503
H    53245
X    40462
touch threshold tau (from 24 peaks per graded minute): 0.51 m/s
possessions: 486 | mean duration: 7.4s | dead share: 43.1%
External reference (Metrica games): ~300 possessions, mean 11-13 s, dead share 35-38%
Saved: /content/drive/MyDrive/Game Theory Barcelona/submissions/possession_submission_v6_touch_external.csv


In [ ]:
# ===== Tracking-only profile: does the Barcelona tracking behave like the external games? =====
# Uses NO ground truth. It only compares how the ball signal and our own output look.
# Needs: cell 2 (EXT data), cell 3 (engines), cell 5 (bx, by, graded_mask, dead_real, labels6).
# Cell 4 (tuning) is NOT required.

def _runs(mask):
    e = np.flatnonzero(np.diff(np.r_[0, mask.astype(np.int8), 0]))
    return (e[1::2] - e[::2]) / FPS                       # run lengths in seconds

def tracking_profile(name, bx, by, graded, dead_mask, labels):
    miss = ~np.isfinite(bx)
    g_min = graded.sum() / FPS / 60
    mr = _runs(miss & graded)                              # runs of missing ball inside graded frames
    dr = _runs(dead_mask & graded)                         # dead stretches we predicted
    lab_g = labels[graded]
    runs_ha = []; s = 0
    for i in range(1, len(lab_g) + 1):
        if i == len(lab_g) or lab_g[i] != lab_g[s]:
            if lab_g[s] in ("H", "A"): runs_ha.append((i - s) / FPS)
            s = i
    q = lambda a, p: float(np.percentile(a, p)) if len(a) else float("nan")
    print(f"{name:12s} | ball missing {miss[graded].mean():5.1%} | missing runs: n/min {len(mr)/g_min:4.1f}, "
          f"median {q(mr,50):4.1f}s, p90 {q(mr,90):5.1f}s, share of runs >=3s {np.mean(mr >= 3) if len(mr) else 0:4.0%}")
    print(f"{'':12s} | dead share {dead_mask[graded].mean():5.1%} | dead stretches {len(dr):3d}, median {q(dr,50):5.1f}s "
          f"| possessions {len(runs_ha):3d}, mean {np.mean(runs_ha):4.1f}s, median {q(runs_ha,50):4.1f}s")

DEAD = dict(miss_min=75, slow_thr=0.0, slow_min=100, close=50, final_min=50)   # same settings as cell 5
for g in (1, 2):
    d = EXT[g]
    if "te" not in d:                                    # built here, so cell 4 is not required
        d["dd"] = DeadDetector(d["bx"], d["by"], pitch=(105.0, 68.0))
        d["te"] = TouchEngine(d["bx"], d["by"], d["dH"], d["dA"], d["graded"], pitch=(105.0, 68.0))
    d["dead_pred"] = d["dd"].mask(**DEAD) & d["graded"]
    lab = d["te"].labels(d["dead_pred"], tau=d["te"].tau_for_rate(24), margin=0.1, flank_gap=25)
    tracking_profile(f"external G{g}", d["bx"], d["by"], d["graded"], d["dead_pred"], lab)
tracking_profile("Barcelona", bx, by, graded_mask, dead_real, labels6)

external G1  | ball missing 37.9% | missing runs: n/min  1.6, median  7.3s, p90  33.9s, share of runs >=3s  58%
             | dead share 38.9% | dead stretches  95, median  18.2s | possessions 302, mean 11.5s, median  6.8s
external G2  | ball missing 40.6% | missing runs: n/min  1.8, median  5.0s, p90  38.7s, share of runs >=3s  57%
             | dead share 40.4% | dead stretches  91, median  18.2s | possessions 267, mean 12.5s, median  8.2s
Barcelona    | ball missing 40.7% | missing runs: n/min  1.4, median 10.7s, p90  33.8s, share of runs >=3s  89%
             | dead share 43.1% | dead stretches 138, median  11.7s | possessions 486, mean  7.4s, median  4.6s


In [ ]:
# ===== Touch-chain profile: why does the Barcelona output have more, shorter possessions? =====
# Uses NO ground truth. Compares the touch chain and the nearest-team signal with the external games.
def touch_chain_profile(name, te, dH, dA, tau, radius=3.0, margin=0.1, flank_gap=25):
    g_min = te.graded.sum() / FPS / 60
    idx = np.flatnonzero((te.dv == te.rollmax) & (te.dv >= tau) & te.graded)
    hh = np.nan_to_num(te.near_h[idx], nan=99.0); aa = np.nan_to_num(te.near_a[idx], nan=99.0)
    ok = (np.minimum(hh, aa) <= radius) & (np.abs(hh - aa) >= margin)
    tf = idx[ok]; tt = np.where(hh[ok] < aa[ok], 0, 1)
    ch_raw = (tt[1:] != tt[:-1]) if len(tt) > 1 else np.array([], bool)
    tf2, tt2 = clean_touches(tf, tt, flank_gap)
    ch_cln = (tt2[1:] != tt2[:-1]) if len(tt2) > 1 else np.array([], bool)
    quick = ch_cln & ((tf2[1:] - tf2[:-1]) < 2 * FPS) if len(tt2) > 1 else np.array([], bool)
    # nearest-team flicker while the ball is within `radius` of a player (graded frames, ball seen)
    seen = np.isfinite(dH) & np.isfinite(dA) & (np.minimum(dH, dA) <= radius) & te.graded
    near_team = np.where(dH < dA, 0, 1)[seen]
    flips = np.sum(near_team[1:] != near_team[:-1]) / g_min if seen.sum() > 1 else float("nan")
    print(f"{name:12s} | peaks kept {len(idx)/g_min:5.1f}/min -> assigned touches {len(tf)/g_min:5.1f}/min "
          f"| team changes/min: raw {ch_raw.sum()/g_min:4.1f}, after cleaning {ch_cln.sum()/g_min:4.1f}, "
          f"within 2 s {quick.sum()/g_min:4.1f} | nearest-team flips near ball {flips:5.1f}/min")

for g in (1, 2):
    d = EXT[g]
    if "te" not in d:
        d["te"] = TouchEngine(d["bx"], d["by"], d["dH"], d["dA"], d["graded"], pitch=(105.0, 68.0))
    touch_chain_profile(f"external G{g}", d["te"], d["dH"], d["dA"], d["te"].tau_for_rate(24))
touch_chain_profile("Barcelona", te_real, dH, dA, tau_real)

external G1  | peaks kept  24.0/min -> assigned touches  23.4/min | team changes/min: raw  3.0, after cleaning  3.0, within 2 s  1.3 | nearest-team flips near ball  13.9/min
external G2  | peaks kept  24.0/min -> assigned touches  23.3/min | team changes/min: raw  2.8, after cleaning  2.7, within 2 s  1.2 | nearest-team flips near ball  14.9/min
Barcelona    | peaks kept  24.0/min -> assigned touches  19.6/min | team changes/min: raw  4.5, after cleaning  4.1, within 2 s  2.3 | nearest-team flips near ball  10.9/min


In [ ]:
gt_pub = pm.label_codes(pm.read_label_csv(os.path.join(SCORE_DIR, "BAR-ATH_possession_gt_public.csv")))
r = pm.evaluate(gt_pub, pm.label_codes(labels6))
print(f"public-window spellF1 {r['spell_f1_mean']:.4f} | macroF1 {r['macro_f1']:.4f} | preds {r['n_pred_spells']} (GT has 17)")

public-window spellF1 0.4190 | macroF1 0.6745 | preds 25 (GT has 17)


In [ ]:
# Optional: download CSV to your laptop
from google.colab import files
files.download(out_csv)